# Explore a captured trace

Loads one trace from a completed run (`llm-interp generate && llm-interp capture`), shows its tokens with think/response/hint/answer spans marked, plots per-layer residual norms, and recomputes one layer's keys from the stored `resid_pre` as a sanity check against a fresh hooked forward pass.

Run `llm-interp generate && llm-interp capture` first if `runs/qwen3_0p6b_local/` doesn't exist yet.

In [ ]:
from pathlib import Path

from transformers import AutoTokenizer

from llm_interp.config import load_config
from llm_interp.loader import Loader

repo_root = Path.cwd()
if not (repo_root / "configs").exists():
    repo_root = repo_root.parent
assert (repo_root / "configs").exists(), f"couldn't find configs/ from {Path.cwd()}"

config = load_config(repo_root / "configs" / "qwen3_0p6b_local.yaml")
tokenizer = AutoTokenizer.from_pretrained(config.model.id, revision=config.model.revision)

storage_root = Path(config.storage_root)
if not storage_root.is_absolute():
    storage_root = repo_root / storage_root

loader = Loader(storage_root, tokenizer)
print(f"{len(loader.trace_ids())} traces available in {storage_root}")

## Load a trace

Picks the first trace_id (by sort order); change `TRACE_ID` to inspect a different one.

In [ ]:
TRACE_ID = sorted(loader.trace_ids())[0]
print("trace_id:", TRACE_ID)
loader.trace_metadata(TRACE_ID)

## Tokens with think/response/hint/answer spans

Spans are half-open `[start, end)` token positions (CLAUDE.md span semantics). A token can fall in more than one span (e.g. `think` and nothing else, since `think`/`response` don't overlap, but `hint` lives in the prompt while everything else lives in the response).

In [ ]:
loaded_tokens = loader.load(TRACE_ID, "token_ids")
spans = {s["span_name"]: (s["start"], s["end"]) for s in loader.spans_for(TRACE_ID)}
print("spans:", spans)


def labels_at(position: int) -> list[str]:
    return [name for name, (start, end) in spans.items() if start <= position < end]


for i, tok in enumerate(loaded_tokens.decoded_tokens):
    tags = labels_at(i)
    marker = f"  <-- {','.join(tags)}" if tags else ""
    print(f"{i:4d} {tok!r:20s}{marker}")

## Per-layer residual norms

Mean `||resid_pre||` at each configured layer, for this trace. Position 0 (the attention sink) is excluded from the mean by default, per CLAUDE.md.

In [ ]:
import matplotlib.pyplot as plt
import torch

layers = config.capture.layers
norms_per_layer = [loader.load(TRACE_ID, "resid_pre", layer=layer).tensor.norm(dim=-1) for layer in layers]
norms_matrix = torch.stack(norms_per_layer)  # [num_layers, N]

# exclude position 0 (attention sink) from the mean
mean_norm_per_layer = norms_matrix[:, 1:].mean(dim=1)

plt.figure(figsize=(8, 4))
plt.plot(layers, mean_norm_per_layer.numpy(), marker="o")
plt.xlabel("layer")
plt.ylabel("mean ||resid_pre|| (excl. position 0)")
plt.title(f"Per-layer residual norm \u2014 trace {TRACE_ID}")
plt.grid(True, alpha=0.3)
plt.show()

## Recompute one layer's keys

Loads a middle layer's stored `resid_pre` and recomputes its post-QK-norm keys via `recompute_qk_pre_rope` (the same function `check_c` validates against the real model) — then, as a sanity check, confirms it matches a fresh hooked forward pass over the same token IDs. This loads the real model, so it's slower than the cells above.

In [ ]:
from llm_interp.hooks import HookManager
from llm_interp.model_loading import build_adapter, load_model_and_tokenizer
from llm_interp.recompute import recompute_qk_pre_rope

model, _ = load_model_and_tokenizer(config)
adapter = build_adapter(model)

chosen_layer = layers[len(layers) // 2]
resid_pre = loader.load(TRACE_ID, "resid_pre", layer=chosen_layer).tensor.unsqueeze(0)  # [1, N, hidden]
_, recomputed_k = recompute_qk_pre_rope(adapter, resid_pre, chosen_layer)
print("recomputed k shape:", recomputed_k.shape)

# ground truth: hook k_norm's own output during a fresh forward pass over the same tokens
token_ids = torch.tensor([loaded_tokens.token_ids]).to(next(model.parameters()).device)
ground_truth = {}
handle = adapter.k_norm(chosen_layer).register_forward_hook(
    lambda module, args, output: ground_truth.setdefault("k", output.detach())
)
try:
    with torch.no_grad():
        model(input_ids=token_ids)
finally:
    handle.remove()

expected_k = ground_truth["k"].transpose(1, 2).to(recomputed_k.device)
print("matches fresh hooked forward pass:", torch.allclose(recomputed_k, expected_k, atol=1e-4, rtol=1e-4))